# Corrección Metodológica: Ensamblado Optimizado en Validación

**Problema detectado:** los pesos del ensamblado se optimizaban mediante grid search **sobre el conjunto de prueba**, el mismo donde luego se reportaba el resultado. Esto introduce un sesgo optimista: el ensamblado ve el test durante su calibración.

**Corrección:** los pesos se optimizan sobre el **conjunto de validación**, y el ensamblado resultante se evalúa sobre el test, que permanece intacto.

| | Antes (sesgado) | Ahora (correcto) |
|---|---|---|
| Optimización de pesos | Conjunto de **prueba** | Conjunto de **validación** |
| Evaluación final | Conjunto de prueba | Conjunto de prueba |
| ¿El test influye en los pesos? | **Sí** (sesgo) | **No** (limpio) |

**Expectativa honesta:** el resultado probablemente baje ligeramente (de 98.75% a ~98.2-98.6%). Ese número menor es el correcto.

Entrena el Top-3 (InceptionV3, VGG16, ConvNeXt-Tiny) y compara ambos métodos para documentar la magnitud del sesgo.

**Tiempo:** ~45 min. GPU T4.

In [ ]:
!pip install -q kaggle timm scikit-learn
import torch
print('GPU:', torch.cuda.is_available())

In [ ]:
from google.colab import files
import os
print('Sube tu kaggle.json:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
for fn in uploaded.keys(): os.rename(fn, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/data --unzip
print('Dataset listo')

In [ ]:
# Misma particion 70/15/15, semilla 42
import shutil, random
from pathlib import Path
random.seed(42)
SRC=Path('/content/data/chest_xray')
ALL={'NORMAL':[],'PNEUMONIA':[]}
for split in ['train','test','val']:
    sd=SRC/split
    if not sd.exists(): continue
    for img in (sd/'NORMAL').glob('*.jpeg'): ALL['NORMAL'].append(img)
    for img in (sd/'PNEUMONIA').glob('*.jpeg'): ALL['PNEUMONIA'].append(img)
DEST=Path('/content/dataset_bin')
if DEST.exists(): shutil.rmtree(DEST)
for cls,imgs in ALL.items():
    imgs=imgs.copy(); random.shuffle(imgs)
    n=len(imgs); ntr=int(n*0.70); nv=int(n*0.15)
    for sn,si in {'train':imgs[:ntr],'val':imgs[ntr:ntr+nv],'test':imgs[ntr+nv:]}.items():
        od=DEST/sn/cls; od.mkdir(parents=True,exist_ok=True)
        for img in si: shutil.copy(img,od/img.name)
for sn in ['train','val','test']:
    print(f'  {sn}:', {c: len(list((DEST/sn/c).glob("*"))) for c in ALL})

In [ ]:
import timm, numpy as np, time, json
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

device=torch.device('cuda')
torch.manual_seed(42); np.random.seed(42)
NUM_CLASSES=2; BATCH=32; EPOCHS=10; LR=1e-4; PATIENCE=4

# El Top-3 real segun la corrida de 9 arquitecturas
TOP3 = {'InceptionV3':('inception_v3',299), 'VGG16':('vgg16',224), 'ConvNeXt-Tiny':('convnext_tiny',224)}

def loaders(img_size):
    train_tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(10),transforms.ColorJitter(brightness=0.15,contrast=0.15),
        transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    eval_tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    tr=datasets.ImageFolder('/content/dataset_bin/train',transform=train_tf)
    va=datasets.ImageFolder('/content/dataset_bin/val',transform=eval_tf)
    te=datasets.ImageFolder('/content/dataset_bin/test',transform=eval_tf)
    return (DataLoader(tr,batch_size=BATCH,shuffle=True,num_workers=2),
            DataLoader(va,batch_size=BATCH,shuffle=False,num_workers=2),
            DataLoader(te,batch_size=BATCH,shuffle=False,num_workers=2))

def get_probs(model, loader):
    model.eval(); probs,labels=[],[]
    with torch.no_grad():
        for x,y in loader:
            probs.extend(torch.softmax(model(x.to(device)),1).cpu().numpy()); labels.extend(y.numpy())
    return np.array(probs), np.array(labels)

def train_model(name, tname, isz):
    tr,va,te = loaders(isz)
    if tname=='inception_v3':
        m=timm.create_model(tname,pretrained=True,num_classes=2,aux_logits=False).to(device)
    else:
        m=timm.create_model(tname,pretrained=True,num_classes=2).to(device)
    crit=nn.CrossEntropyLoss(); opt=optim.AdamW(m.parameters(),lr=LR,weight_decay=0.01)
    sch=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    best,bstate,noimp=0.0,None,0
    print(f'\n--- {name} ---')
    for ep in range(EPOCHS):
        t0=time.time(); m.train()
        for x,y in tr:
            x,y=x.to(device),y.to(device)
            opt.zero_grad(); crit(m(x),y).backward(); opt.step()
        sch.step()
        vp,vl = get_probs(m, va)
        vacc = accuracy_score(vl, vp.argmax(1))
        print(f'  Ep {ep+1}/{EPOCHS} val={vacc:.4f} ({time.time()-t0:.0f}s)')
        if vacc>best: best=vacc; bstate={k:v.cpu().clone() for k,v in m.state_dict().items()}; noimp=0
        else:
            noimp+=1
            if noimp>=PATIENCE: print(f'  Early stop ep {ep+1}'); break
    if bstate: m.load_state_dict(bstate)
    # Probabilidades en VALIDACION y TEST
    pv, lv = get_probs(m, va)
    pt, lt = get_probs(m, te)
    acc_test = accuracy_score(lt, pt.argmax(1))
    print(f'  >> TEST {name}: {acc_test*100:.2f}%')
    del m; torch.cuda.empty_cache()
    return pv, lv, pt, lt, acc_test

In [ ]:
# Entrenar el Top-3 y guardar probabilidades en VAL y TEST
probs_val, probs_test = {}, {}
labels_val = labels_test = None
acc_individual = {}

for name,(tname,isz) in TOP3.items():
    pv, lv, pt, lt, acc = train_model(name, tname, isz)
    probs_val[name]=pv; probs_test[name]=pt
    acc_individual[name]=acc
    if labels_val is None: labels_val=lv; labels_test=lt

print('\nEntrenamiento completo.')
print(f'Validación: {len(labels_val)} imágenes | Test: {len(labels_test)} imágenes')

## Comparación: pesos optimizados en TEST (sesgado) vs. en VALIDACIÓN (correcto)

In [ ]:
from itertools import product
import pandas as pd
names = list(TOP3)

def buscar_pesos(probs, labels):
    """Grid search de pesos que maximiza exactitud sobre el conjunto dado."""
    P = [probs[n] for n in names]
    best_acc, best_w = 0, None
    for w1,w2 in product(np.arange(0,1.05,0.05), repeat=2):
        w3 = 1-w1-w2
        if w3 < -1e-9: continue
        w3 = max(w3, 0)
        ens = w1*P[0] + w2*P[1] + w3*P[2]
        a = accuracy_score(labels, ens.argmax(1))
        if a > best_acc: best_acc, best_w = a, (round(w1,2), round(w2,2), round(w3,2))
    return best_w, best_acc

def evaluar(pesos, probs, labels):
    P = [probs[n] for n in names]
    ens = pesos[0]*P[0] + pesos[1]*P[1] + pesos[2]*P[2]
    pred = ens.argmax(1)
    cm = confusion_matrix(labels, pred); tn,fp = cm[0,0], cm[0,1]
    return {
        'accuracy': accuracy_score(labels, pred),
        'precision': precision_score(labels, pred, average='macro', zero_division=0),
        'recall_sensitivity': recall_score(labels, pred, average='macro', zero_division=0),
        'f1': f1_score(labels, pred, average='macro', zero_division=0),
        'specificity': float(tn/(tn+fp)),
        'confusion_matrix': cm.tolist(),
    }, pred

# MÉTODO A (SESGADO): optimizar en test, reportar en test
w_test, acc_opt_test = buscar_pesos(probs_test, labels_test)
res_sesgado, _ = evaluar(w_test, probs_test, labels_test)

# MÉTODO B (CORRECTO): optimizar en validación, reportar en test
w_val, acc_opt_val = buscar_pesos(probs_val, labels_val)
res_correcto, pred_correcto = evaluar(w_val, probs_test, labels_test)

print('='*60)
print('MÉTODO A — SESGADO (pesos optimizados sobre TEST)')
print('='*60)
print(f'  Pesos {names} = {w_test}')
print(f'  Exactitud en TEST: {res_sesgado["accuracy"]*100:.2f}%  <-- inflado')

print('\n' + '='*60)
print('MÉTODO B — CORRECTO (pesos optimizados sobre VALIDACIÓN)')
print('='*60)
print(f'  Pesos {names} = {w_val}')
print(f'  Exactitud en VALIDACIÓN (usada para elegir pesos): {acc_opt_val*100:.2f}%')
print(f'  Exactitud en TEST (resultado limpio): {res_correcto["accuracy"]*100:.2f}%  <-- ESTE ES EL CORRECTO')

sesgo = (res_sesgado['accuracy'] - res_correcto['accuracy'])*100
print(f'\n>>> SESGO OPTIMISTA CUANTIFICADO: {sesgo:+.2f} puntos porcentuales')

print('\nReporte del ensamblado CORRECTO sobre test:')
print(classification_report(labels_test, pred_correcto, target_names=['NORMAL','PNEUMONIA'], zero_division=0))

In [ ]:
# Guardar
salida = {
    'modelos_individuales_test': {k: v for k,v in acc_individual.items()},
    'metodo_A_sesgado': {'pesos': list(w_test), 'metricas_test': res_sesgado},
    'metodo_B_correcto': {'pesos': list(w_val), 'acc_validacion': float(acc_opt_val), 'metricas_test': res_correcto},
    'sesgo_optimista_puntos': float(sesgo),
    'orden_modelos': names,
}
with open('/content/ensamblado_corregido.json','w') as f:
    json.dump(salida, f, indent=2)

df = pd.DataFrame([
    {'Método':'A. Pesos optimizados en TEST (sesgado)', 'Exactitud test (%)': round(res_sesgado['accuracy']*100,2),
     'Precisión (%)': round(res_sesgado['precision']*100,2), 'Sensibilidad (%)': round(res_sesgado['recall_sensitivity']*100,2),
     'Especificidad (%)': round(res_sesgado['specificity']*100,2), 'F1 (%)': round(res_sesgado['f1']*100,2)},
    {'Método':'B. Pesos optimizados en VALIDACIÓN (correcto)', 'Exactitud test (%)': round(res_correcto['accuracy']*100,2),
     'Precisión (%)': round(res_correcto['precision']*100,2), 'Sensibilidad (%)': round(res_correcto['recall_sensitivity']*100,2),
     'Especificidad (%)': round(res_correcto['specificity']*100,2), 'F1 (%)': round(res_correcto['f1']*100,2)},
])
print(df.to_string(index=False))
df.to_csv('/content/ensamblado_corregido.csv', index=False)

files.download('/content/ensamblado_corregido.json')
files.download('/content/ensamblado_corregido.csv')
print('\nPasame los 2 archivos.')